In [5]:
import yfinance as yf

# ticker=yf.Ticker("AAPL")

# data=ticker.history(period="5d")

# print(data)
# print(type(data))
data = yf.download(
    tickers="TSLA",
    start="2020-01-01",
    end="2025-12-31",
    interval="1d"

)

print(data.head())
print(data.tail())
print(data.shape)


/tmp/ipython-input-2422557447.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           TSLA       TSLA       TSLA       TSLA       TSLA
Date                                                             
2020-01-02  28.684000  28.713333  28.114000  28.299999  142981500
2020-01-03  29.534000  30.266666  29.128000  29.366667  266677500
2020-01-06  30.102667  30.104000  29.333332  29.364668  151995000
2020-01-07  31.270666  31.441999  30.224001  30.760000  268231500
2020-01-08  32.809334  33.232666  31.215334  31.580000  467164500
Price            Close        High         Low        Open    Volume
Ticker            TSLA        TSLA        TSLA        TSLA      TSLA
Date                                                                
2025-12-23  485.559998  491.970001  482.839996  489.399994  58223600
2025-12-24  485.399994  490.899994  476.799988  488.480011  41285400
2025-12-26  475.190002  489.089996  473.820007  485.230011  58780700
2025-12-29  459.640015  469.399994  459.000000  469.000000

In [6]:
import sqlite3

conn=sqlite3.connect("trading_data.db")

cursor=conn.cursor()

cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS stock_prices(
     ticker TEXT,
     date TEXT,
     open REAL,
     high REAL,
     low REAL,
     close REAL,
     volume INTEGER
    )
""")

conn.commit()
conn.close()

print("Database created successfully")


Database created successfully


In [8]:
import sqlite3
import pandas as pd

conn=sqlite3.connect("trading_data.db")

df=pd.read_sql("SELECT * FROM stock_prices LIMIT 10",conn)
print(df)

count=pd.read_sql("SELECT COUNT(*) FROM stock_prices",conn)
print(f"\nTotal rows in database: {count.iloc[0,0]}")

conn.close()

         Date      Close       High        Low       Open     Volume Ticker
0  2020-01-02  72.400520  72.460784  71.156682  71.409785  135480400   AAPL
1  2020-01-03  71.696640  72.455958  71.472462  71.629145  146322800   AAPL
2  2020-01-06  72.267937  72.306506  70.568510  70.819208  118387200   AAPL
3  2020-01-07  71.928055  72.533095  71.708695  72.277578  108872000   AAPL
4  2020-01-08  73.085114  73.386431  71.631559  71.631559  132079200   AAPL
5  2020-01-09  74.637497  74.830337  73.810684  74.061375  170108400   AAPL
6  2020-01-10  74.806229  75.370301  74.304840  74.871318  140644800   AAPL
7  2020-01-13  76.404404  76.430923  75.003882  75.122003  121532000   AAPL
8  2020-01-14  75.372719  76.551476  75.249786  76.341760  161954400   AAPL
9  2020-01-15  75.049698  76.052483  74.618209  75.172638  121923600   AAPL

Total rows in database: 2012


In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("trading_data.db")

# See ALL column names in the table
df = pd.read_sql("SELECT * FROM stock_prices LIMIT 3", conn)
print("Columns:", df.columns.tolist())
print(df)

conn.close()

Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
         Date      Close       High        Low       Open     Volume Ticker
0  2020-01-02  72.400520  72.460784  71.156682  71.409785  135480400   AAPL
1  2020-01-03  71.696640  72.455958  71.472462  71.629145  146322800   AAPL
2  2020-01-06  72.267937  72.306506  70.568510  70.819208  118387200   AAPL


In [13]:
import yfinance as yf
import sqlite3
import pandas as pd
import numpy as np


DB_NAME="trading_data.db"

def save_stock_data(ticker,start_date,end_date):
  data=yf.download(ticker,start=start_date,end=end_date)
  if data.empty:
    print(f"error data empty")
    return
  data.columns=data.columns.get_level_values(0)

  data=data.reset_index()
  data["Date"]=data["Date"].dt.strftime("%Y-%m-%d")
  data["Ticker"]=ticker

  conn=sqlite3.connect(DB_NAME)
  data.to_sql("stock_prices",conn,if_exists="append",index=False)
  conn.close()

  print(f"saved {len(data)} rows for {ticker}")

def get_stock_data(ticker,start_date,end_date):
  conn=sqlite3.connect(DB_NAME)
  query="""
        SELECT * FROM stock_prices
        WHERE Ticker = ?
        AND Date >= ?
        AND Date <= ?
        ORDER BY Date
    """
  df=pd.read_sql(query,conn,params=[ticker,start_date,end_date])
  conn.close()
  return df




save_stock_data("AAPL","2020-01-01","2025-12-31")
save_stock_data("MSFT","2020-01-01","2025-12-31")
save_stock_data("TSLA","2020-01-01","2025-12-31")

conn=sqlite3.connect(DB_NAME)

# df=pd.read_sql("SELECT * FROM stock_prices LIMIT 3",conn)
# print("Columns: ",df.columns.tolist())
# print(df)
# conn.close()

df=get_stock_data("AAPL","2020-01-01","2025-12-31")
# print(df.head())
# print(f"got {len(df)} rows")

df["Daily_Return"]=df["Close"].pct_change()
# Calculate daily returns
print(df[["Date", "Close", "Daily_Return"]].head(10))
#typical daily move
print(f"Average daily return : {df['Daily_Return'].mean(): .4f}")
#highest single day gain
print(f"Best day: {df['Daily_Return'].max(): .4f}")
#highest single day loss
print(f"Biggest loss: {df['Daily_Return'].min(): .4f}")

#daily votality= standard deviation of daily returns
daily_vol=df["Daily_Return"].std()

# Annualized volatility (multiply by sqrt of trading days in a year)
# There are ~252 trading days per year

annual_vol=daily_vol *np.sqrt(252)

print(f"Daily volatility: {daily_vol: .4f}")
print(f"Annual volatility: {annual_vol:.4f}")
print(f"Annual volatility: {annual_vol*100:.1f}%")

# This is a KEY concept from statistics.

# If daily standard deviation = σ
# Then over N days, standard deviation = σ × √N

# There are 252 trading days in a year.
# So annual volatility = daily_vol × √252

# This lets you COMPARE stocks on a yearly basis.

# Example:
#   AAPL annual vol = 30%  → "In a typical year, AAPL moves about 30%"
#   TSLA annual vol = 60%  → "TSLA is TWICE as wild"


# Step 1: Calculate the running peak (highest price seen SO FAR)
df["Peak"]=df["Close"].cummax()

# Step 2: Calculate drawdown at each point
df["Drawdown"]=(df["Close"]-df["Peak"])/df["Peak"]

# Step 3: Find the worst (minimum) drawdown
max_drawdown=df["Drawdown"].min()

print(f"Max Drawdown: {max_drawdown:.4f}")
print(f"Max Drawdown: {max_drawdown*100: .1f}%")

# Show the date it happened
worst_date=df.loc[df["Drawdown"].idxmin(),"Date"]
print(f"Worst point : {worst_date}")











/tmp/ipython-input-2062503066.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data=yf.download(ticker,start=start_date,end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-2062503066.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data=yf.download(ticker,start=start_date,end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-2062503066.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data=yf.download(ticker,start=start_date,end=end_date)
[*********************100%***********************]  1 of 1 completed


saved 1507 rows for AAPL
saved 1507 rows for MSFT
saved 1507 rows for TSLA
         Date      Close  Daily_Return
0  2020-01-02  72.400520           NaN
1  2020-01-02  72.400490 -4.215105e-07
2  2020-01-02  72.400490  0.000000e+00
3  2020-01-03  71.696640 -9.721616e-03
4  2020-01-03  71.696632 -1.064122e-07
5  2020-01-03  71.696632  0.000000e+00
6  2020-01-06  72.267937  7.968356e-03
7  2020-01-06  72.267921 -2.111419e-07
8  2020-01-06  72.267921  0.000000e+00
9  2020-01-07  71.928055 -4.702870e-03
Average daily return :  0.0004
Best day:  0.1533
Biggest loss: -0.1286
Daily volatility:  0.0123
Annual volatility: 0.1949
Annual volatility: 19.5%
Max Drawdown: -0.3336
Max Drawdown: -33.4%
Worst point : 2025-04-08


In [19]:
import yfinance as yf
import numpy as np
import pandas as pd
import sqlite3

DB_NAME="trading_data.db"

def remove_duplicates():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        DELETE FROM stock_prices
        WHERE rowid NOT IN (
            SELECT MIN(rowid)
            FROM stock_prices
            GROUP BY Ticker, Date
        );
    """)

    conn.commit()
    conn.close()
    print("Duplicate rows removed")



def save_stock_data(ticker,start_date,end_date):
  data=yf.download(ticker,start=start_date,end=end_date)
  if data.empty:
    print(f"error data empty")
    return
  data.columns=data.columns.get_level_values(0)

  data=data.reset_index()
  data["Date"]=data["Date"].dt.strftime("%Y-%m-%d")
  data["Ticker"]=ticker

  conn=sqlite3.connect(DB_NAME)
  try:
    data.to_sql("stock_prices", conn, if_exists="append", index=False)
  except Exception as e:
    print("Duplicate rows skipped")
  conn.close()

  print(f"saved {len(data)} rows for {ticker}")

def create_unique_index():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_unique_stock
        ON stock_prices (Ticker, Date);
    """)

    conn.commit()
    conn.close()
remove_duplicates()
create_unique_index()


save_stock_data("AAPL","2020-01-01","2025-12-31")
save_stock_data("MSFT","2020-01-01","2025-12-31")
save_stock_data("TSLA","2020-01-01","2025-12-31")



def get_stock_data(ticker,start_date,end_date):
  conn=sqlite3.connect(DB_NAME)
  query="""
      SELECT * FROM stock_prices
      WHERE Ticker = ?
      AND Date >= ?
      AND Date <= ?
      ORDER BY Date
  """
  df=pd.read_sql(query,conn,params=[ticker,start_date,end_date])
  conn.close()
  return df

def calculate_stats(ticker,start_date,end_date):
  """
  Calculate returns ,volatility, max drawdown for a stock.
  """

  df=get_stock_data(ticker,start_date,end_date)

  if df.empty:
    print(f"no data")
    return None
  #RETURNS
  df["Daily_Return"]=df["Close"].pct_change()
  avg_daily_return=df["Daily_Return"].mean()
  total_return=(df["Close"].iloc[-1]-df["Close"].iloc[0])/df["Close"].iloc[0]

  #VOLATILITY
  daily_vol=df["Daily_Return"].std()
  annual_vol=daily_vol *np.sqrt(252)


  #MAX DRAWDOWN

  df["Peak"]=df["Close"].cummax()
  df["Drawdown"]=(df["Close"]-df["Peak"])/df["Peak"]
  max_drawdown=df["Drawdown"].min()
  worst_date=df.loc[df["Drawdown"].idxmin(),"Date"]

  #print report
  print(f"\n{'='*50}")
  print(f"📊 STATS FOR {ticker}")
  print(f"📅 {start_date} to {end_date}")

  print(f"\n{'='*50}")
  print(f"  Total Return:      {total_return*100:>8.1f}%")
  print(f"  Avg Daily Return:  {avg_daily_return*100:>8.4f}%")
  print(f"  Daily Volatility:  {daily_vol*100:>8.4f}%")
  print(f"  Annual Volatility: {annual_vol*100:>8.1f}%")
  print(f"  Max Drawdown:      {max_drawdown*100:>8.1f}%")
  print(f"  Best Day:          {df['Daily_Return'].max()*100:>8.2f}%")
  print(f"  Worst Day:         {df['Daily_Return'].min()*100:>8.2f}%")
  print(f"  Total Trading Days: {len(df)}")
  print(f"{'='*50}\n")

  return None


# df=calculate_stats("AAPL", "2020-01-01", "2025-12-31")

calculate_stats("AAPL", "2020-01-01", "2025-12-31")
calculate_stats("MSFT", "2020-01-01", "2025-12-31")
calculate_stats("TSLA", "2020-01-01", "2025-12-31")







/tmp/ipython-input-3552966733.py:28: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data=yf.download(ticker,start=start_date,end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3552966733.py:28: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data=yf.download(ticker,start=start_date,end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3552966733.py:28: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data=yf.download(ticker,start=start_date,end=end_date)
[*********************100%***********************]  1 of 1 completed

Duplicate rows removed
Duplicate rows skipped
saved 1507 rows for AAPL
Duplicate rows skipped
saved 1507 rows for MSFT
Duplicate rows skipped
saved 1507 rows for TSLA

📊 STATS FOR AAPL
📅 2020-01-01 to 2025-12-31

  Total Return:         276.8%
  Avg Daily Return:    0.1081%
  Daily Volatility:    2.0044%
  Annual Volatility:     31.8%
  Max Drawdown:         -33.4%
  Best Day:             15.33%
  Worst Day:           -12.86%
  Total Trading Days: 1507


📊 STATS FOR MSFT
📅 2020-01-01 to 2025-12-31

  Total Return:         219.6%
  Avg Daily Return:    0.0945%
  Daily Volatility:    1.8618%
  Annual Volatility:     29.6%
  Max Drawdown:         -37.1%
  Best Day:             14.22%
  Worst Day:           -14.74%
  Total Trading Days: 1507


📊 STATS FOR TSLA
📅 2020-01-01 to 2025-12-31

  Total Return:        1484.3%
  Avg Daily Return:    0.2712%
  Daily Volatility:    4.1948%
  Annual Volatility:     66.6%
  Max Drawdown:         -73.6%
  Best Day:             22.69%
  Worst Day:       

In [20]:
import yfinance as yf
import numpy as np
import pandas as pd
import sqlite3

DB_NAME = "trading_data.db"


def remove_duplicates():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        DELETE FROM stock_prices
        WHERE rowid NOT IN (
            SELECT MIN(rowid)
            FROM stock_prices
            GROUP BY Ticker, Date
        );
    """)

    conn.commit()
    conn.close()
    print("Duplicate rows removed")


def create_unique_index():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_unique_stock
        ON stock_prices (Ticker, Date);
    """)

    conn.commit()
    conn.close()


def save_stock_data(ticker, start_date, end_date):
    data = yf.download(ticker, start=start_date, end=end_date)
    if data.empty:
        print(f"error data empty")
        return

    data.columns = data.columns.get_level_values(0)
    data = data.reset_index()
    data["Date"] = data["Date"].dt.strftime("%Y-%m-%d")
    data["Ticker"] = ticker

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    inserted = 0
    skipped = 0

    for _, row in data.iterrows():
        try:
            cursor.execute("""
                INSERT INTO stock_prices (Date, Open, High, Low, Close, Volume, Ticker)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (row["Date"], row["Open"], row["High"], row["Low"],
                  row["Close"], row["Volume"], row["Ticker"]))
            inserted += 1
        except sqlite3.IntegrityError:
            skipped += 1

    conn.commit()
    conn.close()

    print(f"✅ {ticker}: {inserted} new rows saved, {skipped} duplicates skipped")


def get_stock_data(ticker, start_date, end_date):
    conn = sqlite3.connect(DB_NAME)
    query = """
        SELECT * FROM stock_prices
        WHERE Ticker = ?
        AND Date >= ?
        AND Date <= ?
        ORDER BY Date
    """
    df = pd.read_sql(query, conn, params=[ticker, start_date, end_date])
    conn.close()
    return df


def calculate_stats(ticker, start_date, end_date):
    """
    Calculate returns, volatility, max drawdown for a stock.
    """
    df = get_stock_data(ticker, start_date, end_date)

    if df.empty:
        print(f"no data")
        return None

    # RETURNS
    df["Daily_Return"] = df["Close"].pct_change()
    avg_daily_return = df["Daily_Return"].mean()
    total_return = (df["Close"].iloc[-1] - df["Close"].iloc[0]) / df["Close"].iloc[0]

    # VOLATILITY
    daily_vol = df["Daily_Return"].std()
    annual_vol = daily_vol * np.sqrt(252)

    # MAX DRAWDOWN
    df["Peak"] = df["Close"].cummax()
    df["Drawdown"] = (df["Close"] - df["Peak"]) / df["Peak"]
    max_drawdown = df["Drawdown"].min()
    worst_date = df.loc[df["Drawdown"].idxmin(), "Date"]

    # print report
    print(f"\n{'='*50}")
    print(f"📊 STATS FOR {ticker}")
    print(f"📅 {start_date} to {end_date}")
    print(f"{'='*50}")
    print(f"  Total Return:      {total_return*100:>8.1f}%")
    print(f"  Avg Daily Return:  {avg_daily_return*100:>8.4f}%")
    print(f"  Daily Volatility:  {daily_vol*100:>8.4f}%")
    print(f"  Annual Volatility: {annual_vol*100:>8.1f}%")
    print(f"  Max Drawdown:      {max_drawdown*100:>8.1f}%")
    print(f"  Worst Drawdown On: {worst_date}")
    print(f"  Best Day:          {df['Daily_Return'].max()*100:>8.2f}%")
    print(f"  Worst Day:         {df['Daily_Return'].min()*100:>8.2f}%")
    print(f"  Total Trading Days: {len(df)}")
    print(f"{'='*50}\n")

    return df


# ============ MAIN ============

if __name__ == "__main__":
    remove_duplicates()
    create_unique_index()

    save_stock_data("AAPL", "2020-01-01", "2025-12-31")
    save_stock_data("MSFT", "2020-01-01", "2025-12-31")
    save_stock_data("TSLA", "2020-01-01", "2025-12-31")

    calculate_stats("AAPL", "2020-01-01", "2025-12-31")
    calculate_stats("MSFT", "2020-01-01", "2025-12-31")
    calculate_stats("TSLA", "2020-01-01", "2025-12-31")

/tmp/ipython-input-3317169743.py:41: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Duplicate rows removed



/tmp/ipython-input-3317169743.py:41: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

✅ AAPL: 0 new rows saved, 1507 duplicates skipped



/tmp/ipython-input-3317169743.py:41: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

✅ MSFT: 0 new rows saved, 1507 duplicates skipped


✅ TSLA: 0 new rows saved, 1507 duplicates skipped

📊 STATS FOR AAPL
📅 2020-01-01 to 2025-12-31
  Total Return:         276.8%
  Avg Daily Return:    0.1081%
  Daily Volatility:    2.0044%
  Annual Volatility:     31.8%
  Max Drawdown:         -33.4%
  Worst Drawdown On: 2025-04-08
  Best Day:             15.33%
  Worst Day:           -12.86%
  Total Trading Days: 1507


📊 STATS FOR MSFT
📅 2020-01-01 to 2025-12-31
  Total Return:         219.6%
  Avg Daily Return:    0.0945%
  Daily Volatility:    1.8618%
  Annual Volatility:     29.6%
  Max Drawdown:         -37.1%
  Worst Drawdown On: 2022-11-03
  Best Day:             14.22%
  Worst Day:           -14.74%
  Total Trading Days: 1507


📊 STATS FOR TSLA
📅 2020-01-01 to 2025-12-31
  Total Return:        1484.3%
  Avg Daily Return:    0.2712%
  Daily Volatility:    4.1948%
  Annual Volatility:     66.6%
  Max Drawdown:         -73.6%
  Worst Drawdown On: 2023-01-03
  Best Day:             22.69%
  Worst Day:           -21.06%
  Total Trad